In [ ]:
import shutil
import subprocess

import yt_dlp
from yt_dlp.utils import DownloadError

def _build_js_runtime_opts():
    """
    yt-dlp 2025.11.12 起，YouTube 需要「外部 JS runtime + yt-dlp-ejs」才能正常解析。
    預設只啟用 deno；若要用 Node.js 必須明確指定，且 Node 需 >= 22。
    """
    runtimes = {}

    if shutil.which("deno"):
        runtimes["deno"] = {}

    node_path = shutil.which("node")
    if node_path:
        try:
            ver = subprocess.run([node_path, "--version"], capture_output=True, text=True).stdout.strip()
            major = int(ver.lstrip("v").split(".")[0])
            if major >= 22:
                runtimes["node"] = {"path": node_path}
            else:
                print(f"⚠️ Node.js 版本 {ver} 太舊，yt-dlp 需要 Node >= 22，請升級。")
        except Exception:
            print("⚠️ 無法判斷 Node.js 版本。")

    if not runtimes:
        print("⚠️ 找不到可用的 JS runtime（deno 或 node>=22），YouTube 可能無法下載。")

    opts = {"js_runtimes": runtimes} if runtimes else {}

    # 若沒有安裝 yt-dlp-ejs，允許從 GitHub 下載 EJS 腳本作為備援
    try:
        import yt_dlp_ejs  # noqa: F401
    except ImportError:
        print("⚠️ 未安裝 yt-dlp-ejs，改用遠端元件（建議: pip install -U \"yt-dlp[default]\"）")
        opts["remote_components"] = ["ejs:github"]

    return opts


def download_1080p_video(url: str) -> None:
    """
    下載 YouTube 影片（最高 1080p，最終輸出 MP4）。

    - 優先 AV1 -> H.264 -> 其他編碼，全部限制 <=1080p
    - 強制輸出容器為 MP4
    - 縮圖轉 JPG 後嵌入 MP4
    - Windows 安全檔名（restrictfilenames=True）

    環境需求：
    - ffmpeg 已在 PATH
    - pip install -U "yt-dlp[default]" mutagen
    - deno 或 Node.js >= 22
    """

    def progress_hook(d):
        status = d.get("status")
        if status == "downloading":
            if not progress_hook._analyzing_printed:
                print("Downloading...")
                progress_hook._analyzing_printed = True
        elif status == "finished":
            print("Merging...")

    progress_hook._analyzing_printed = False

    ydl_opts = {
        # 原本 AV1 找不到時會直接掉到 best（常常只有 360p），這裡加入中間的回退層級
        "format": (
            "bestvideo[height<=1080][vcodec^=av01]+bestaudio[ext=m4a]/"
            "bestvideo[height<=1080][vcodec^=avc1]+bestaudio[ext=m4a]/"
            "bestvideo[height<=1080]+bestaudio/"
            "best[height<=1080]/best"
        ),
        "merge_output_format": "mp4",
        "outtmpl": "%(title)s.%(ext)s",
        "restrictfilenames": True,
        "writethumbnail": True,
        "postprocessors": [
            {
                # 用 remux（不重新編碼）取代 convert，速度快且不損畫質
                "key": "FFmpegVideoRemuxer",
                "preferedformat": "mp4",
            },
            {
                "key": "FFmpegMetadata",
                "add_metadata": True,
                "add_chapters": True,
            },
            {
                "key": "FFmpegThumbnailsConvertor",
                "format": "jpg",
                "when": "before_dl",
            },
            {
                "key": "EmbedThumbnail",
                "already_have_thumbnail": False,
            },
        ],
        "progress_hooks": [progress_hook],
        "noplaylist": True,
        "quiet": False,
        "no_warnings": False,
    }
    ydl_opts.update(_build_js_runtime_opts())

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        print("Done")
    except DownloadError as e:
        print(f"Download failed: {e}")
        print('💡 請先 pip install -U "yt-dlp[default]"，並安裝 deno 或 Node.js >= 22。')


if __name__ == "__main__":
    url = input("Enter YouTube URL: ").strip()
    download_1080p_video(url)
